![Traffic](traffic.png)

Traffic data fluctuates constantly or is affected by time. Predicting it can be challenging, but this task will help sharpen your time-series skills. With deep learning, you can use abstract patterns in data that can help boost predictability.

Your task is to build a system that can be applied to help you predict traffic volume or the number of vehicles passing at a specific point and time. Determining this can help reduce road congestion, support new designs for roads or intersections, improve safety, and more! Or, you can use to help plan your commute to avoid traffic!

The dataset provided contains the hourly traffic volume on an interstate highway in Minnesota, USA. It also includes weather features and holidays, which often impact traffic volume.

Time to predict some traffic!

### The data:

The dataset is collected and maintained by UCI Machine Learning Repository. The target variable is `traffic_volume`. The dataset contains the following and has already been normalized and saved into training and test sets:

`train_scaled.csv`, `test_scaled.csv`
| Column     | Type       | Description              |
|------------|------------|--------------------------|
|`temp`                   |Numeric            |Average temp in kelvin|
|`rain_1h`                |Numeric            |Amount in mm of rain that occurred in the hour|
|`snow_1h`                |Numeric            |Amount in mm of snow that occurred in the hour|
|`clouds_all`             |Numeric            |Percentage of cloud cover|
|`date_time`              |DateTime           |Hour of the data collected in local CST time|
|`holiday_` (11 columns)  |Categorical        |US National holidays plus regional holiday, Minnesota State Fair|
|`weather_main_` (11 columns)|Categorical     |Short textual description of the current weather|
|`weather_description_` (35 columns)|Categorical|Longer textual description of the current weather|
|`hour_of_day`|Numeric|The hour of the day|
|`day_of_week`|Numeric|The day of the week (0=Monday, Sunday=6)|
|`day_of_month`|Numeric|The day of the month|
|`month`|Numeric|The number of the month|
|`traffic_volume`         |Numeric            |Hourly I-94 ATR 301 reported westbound traffic volume|

In [4]:
# Import the relevant libraries
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [5]:
# Read the traffic data from the CSV training and test files
train_scaled_df = pd.read_csv('train_scaled.csv')
test_scaled_df = pd.read_csv('test_scaled.csv')

# Convert the DataFrame to NumPy arrays
train_scaled = train_scaled_df.to_numpy()
test_scaled = test_scaled_df.to_numpy()

In [6]:
# Create sequences for time-series forecasting
def create_sequences(data, sequence_length, target_col_idx):
    inputs = []
    targets = []

    for i in range(len(data) - sequence_length):
        sequence = data[i:i + sequence_length]
        target = data[i + sequence_length][target_col_idx]

        inputs.append(sequence)
        targets.append(target)

    return np.array(inputs), np.array(targets)


# Use the previous 24 hours to predict the next hour
sequence_length = 24

# Find the position of the traffic_volume column
target_col_idx = train_scaled_df.columns.get_loc("traffic_volume")

# Create training and testing sequences
X_train, y_train = create_sequences(
    train_scaled,
    sequence_length,
    target_col_idx
)

X_test, y_test = create_sequences(
    test_scaled,
    sequence_length,
    target_col_idx
)


# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(
    X_train.astype(np.float32)
).float()

y_train_tensor = torch.tensor(
    y_train.astype(np.float32)
).float()

X_test_tensor = torch.tensor(
    X_test.astype(np.float32)
).float()

y_test_tensor = torch.tensor(
    y_test.astype(np.float32)
).float()


# Combine features and targets into datasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


# Define the LSTM neural network
class TrafficModel(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.activation = nn.ReLU()

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        # Pass the sequences through the LSTM
        _, (h_n, _) = self.lstm(x)

        # Take the final hidden state
        x = h_n[-1]

        # Apply activation
        x = self.activation(x)

        # Produce one traffic-volume prediction
        x = self.fc(x)

        return x


# Initialize the model
input_size = X_train_tensor.shape[2]

traffic_model = TrafficModel(
    input_size=input_size,
    hidden_size=64,
    num_layers=2
)


# Mean Squared Error loss for regression
criterion = nn.MSELoss()


# Adam optimizer
optimizer = optim.Adam(
    traffic_model.parameters(),
    lr=0.001
)


# Train the model
num_epochs = 2

for epoch in range(num_epochs):

    traffic_model.train()

    for X_batch, y_batch in train_loader:

        # Clear gradients from the previous step
        optimizer.zero_grad()

        # Make predictions
        predictions = traffic_model(X_batch).squeeze(1)

        # Calculate loss
        loss = criterion(
            predictions,
            y_batch
        )

        # Calculate gradients
        loss.backward()

        # Update model weights
        optimizer.step()

    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Loss: {loss.item():.4f}"
    )


# Save the final training loss as a tensor
final_training_loss = loss.detach()


# Evaluate on the test set
traffic_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        predictions = traffic_model(X_batch).squeeze(1)

        all_predictions.append(predictions)

        all_labels.append(y_batch)


# Combine all batches
all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)


# Calculate test Mean Squared Error
test_mse = F.mse_loss(
    all_predictions,
    all_labels
)


print("Final training loss:", final_training_loss)
print("Test MSE:", test_mse)

Epoch 1/2, Loss: 0.0173
Epoch 2/2, Loss: 0.0047
Final training loss: tensor(0.0047)
Test MSE: tensor(0.0024)
